In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import finlab
from finlab import data
from finlab.dataframe import FinlabDataFrame


class StockAnalyzer:
    def __init__(self, excel_path='select_stock.xlsx', days=240, output_dir='output'):
        # Step 1: 初始化基本參數與登入
        self.excel_path = excel_path
        self.days = days
        self.ma_periods = [5, 20, 60, 120]
        self.ma_ckeys = ['ma5', 'ma20', 'ma60', 'ma120']
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        finlab.login(os.environ.get('FINLAB_API_KEY'))

        # Step 2: 配色設定 (大戶紅、散戶綠)
        self.colors = {
            'ma5':   '#F39C12',
            'ma20':  '#2E86C1',
            'ma60':  '#8E44AD',
            'ma120': '#717D7E',
            'h260':  '#E74C3C',
            'cond_active': {
                '創260高':   '#E74C3C',
                '低波動':    '#27AE60',
                '融資健康':  '#2E86C1',
                '營收9月高': '#F39C12',
                '投信買超':  '#8E44AD',
            },
            'cond_inactive': 'rgba(180,180,180,0.20)',
            'yoy_pos':  '#C0392B',
            'yoy_neg':  '#27AE60',
            'mom_line': '#E67E22',
            'inv_big':  '#E74C3C',
            'inv_ppl':  '#27AE60',
            'grid':     'rgba(200,200,200,0.35)',
            'bg':       '#FAFBFC',
            'panel_bg': 'white',
        }

        self.stock_ids = self._load_stock_ids()
        self._fetch_and_prepare_data()

    def _load_stock_ids(self):
        seen = set()
        stock_ids = []
        self.stock_strategies = {}  # {stock_id: [策略名稱]}
        if os.path.exists(self.excel_path):
            df_sel = pd.read_excel(self.excel_path)
            for col in df_sel.columns:
                for x in df_sel[col].dropna():
                    try:
                        sid = str(int(x))
                        if sid not in seen:
                            seen.add(sid)
                            stock_ids.append(sid)
                        self.stock_strategies.setdefault(sid, [])
                        if col not in self.stock_strategies[sid]:
                            self.stock_strategies[sid].append(col)
                    except (ValueError, TypeError):
                        pass
        print('選股清單：', stock_ids)
        return stock_ids

    def _fetch_and_prepare_data(self):
        print("資料下載與運算中...")
        self.close = data.get('price:收盤價')
        self.open_ = data.get('price:開盤價')
        self.high = data.get('price:最高價')
        self.low = data.get('price:最低價')
        self.volume = data.get('price:成交股數')
        self.margin = data.get('margin_transactions:融資使用率').fillna(0)
        self.rev = data.get('monthly_revenue:當月營收')
        self.yoy = data.get('monthly_revenue:去年同月增減(%)')
        self.mom = data.get('monthly_revenue:上月比較增減(%)')
        self.trust = data.get(
            'institutional_investors_trading_summary:投信買賣超股數') / 1000
        self.inv = data.get('inventory')

        (self.INV_SYM, self.INV_DATE, self.INV_BUCKET,
         self.INV_PERSON, self.INV_SHARES, self.INV_RATIO) = self.inv.columns[:6]

        level = self.inv['持股分級'].astype(int)
        h1 = FinlabDataFrame(
            self.inv[level <= 4].pivot_table(
                index='date', columns='stock_id', values='持有股數',
                aggfunc='sum', observed=False
            )
        )
        h2 = FinlabDataFrame(
            self.inv[(level >= 11) & (level <= 15)].pivot_table(
                index='date', columns='stock_id', values='持有股數',
                aggfunc='sum', observed=False
            )
        )
        self.inv_pr_rank = (h2 / (h1 + h2)).diff(6).rank(axis=1,
                                                         pct=True) * self.close.notna()
        self.sid_exist = [s for s in self.stock_ids if s in self.close.columns]

        # 公司名稱對照表
        try:
            co = data.get('company_basic_info')
            self.company_map = dict(zip(co['stock_id'].astype(str), co['公司簡稱']))
        except Exception:
            self.company_map = {}

    def _inv_stats(self, sid, daily_idx):
        nan_s = pd.Series(0.0, index=daily_idx)
        mask = self.inv[self.INV_SYM].astype(str) == str(sid)
        if not mask.any():
            return nan_s, nan_s

        sub = self.inv[mask]
        level = sub[self.INV_BUCKET].astype(int)
        big_sub   = sub[(level >= 11) & (level <= 15)]
        small_sub = sub[(level >= 1)  & (level <= 4)]

        # 週頻持股比 diff(1)，只在集保更新日有值
        # reindex 不 ffill → 非更新日為 NaN → fillna(0) → 每週只顯示一根柱
        big_ratio   = big_sub.groupby(self.INV_DATE)[self.INV_RATIO].sum()
        small_ratio = small_sub.groupby(self.INV_DATE)[self.INV_RATIO].sum()
        big_chg_w   = big_ratio.diff(1)
        small_chg_w = small_ratio.diff(1)

        return (big_chg_w.reindex(daily_idx).ffill().fillna(0),
                small_chg_w.reindex(daily_idx).ffill().fillna(0))


    def _build_fig(self, sid):
        """建立單一股票的 Plotly figure，回傳 (fig, date_str, summary_text)"""
        c_all = self.close[sid].dropna()
        buf = c_all.tail(self.days + 260)
        c = buf.tail(self.days)
        idx = c.index

        o = self.open_[sid].reindex(idx) if sid in self.open_.columns else pd.Series(np.nan, index=idx)
        h = self.high[sid].reindex(idx) if sid in self.high.columns else pd.Series(np.nan, index=idx)
        l = self.low[sid].reindex(idx) if sid in self.low.columns else pd.Series(np.nan, index=idx)
        v = self.volume[sid].reindex(idx) if sid in self.volume.columns else pd.Series(np.nan, index=idx)

        ma_dict = {p: buf.rolling(p).mean().reindex(idx) for p in self.ma_periods}
        high_260 = buf.rolling(260, min_periods=130).max().reindex(idx)

        o_buf = self.open_[sid].reindex(buf.index) if sid in self.open_.columns else buf
        h_buf = self.high[sid].reindex(buf.index) if sid in self.high.columns else buf
        l_buf = self.low[sid].reindex(buf.index) if sid in self.low.columns else buf
        is_up = buf >= o_buf
        bv_up = (abs(buf.shift()-o_buf)+abs(o_buf-l_buf)+abs(l_buf-h_buf)+abs(h_buf-buf))
        bv_dn = (abs(buf.shift()-o_buf)+abs(o_buf-h_buf)+abs(h_buf-l_buf)+abs(l_buf-buf))
        candle_vol = ((bv_up.where(is_up, bv_dn).rolling(20).mean() / buf.rolling(20).mean() * 100).reindex(idx))

        margin_d = (self.margin[sid].reindex(idx, method='ffill').fillna(0)
                    if sid in self.margin.columns else pd.Series(0.0, index=idx))
        margin_ok = (margin_d > 2) & (margin_d < 40)

        if sid in self.rev.columns:
            rev_s = self.rev[sid].dropna()
            rev_ma2 = rev_s.rolling(2, min_periods=1).mean()
            rev_9max = rev_ma2.rolling(9, min_periods=6).max()
            rev_daily = (rev_ma2 >= rev_9max * 0.999).reindex(idx, method='ffill').fillna(False)
        else:
            rev_daily = pd.Series(False, index=idx)

        yoy_d = (self.yoy[sid].reindex(idx, method='ffill').fillna(0)
                 if sid in self.yoy.columns else pd.Series(0.0, index=idx))
        mom_d = (self.mom[sid].reindex(idx, method='ffill').fillna(0)
                 if sid in self.mom.columns else pd.Series(0.0, index=idx))

        if sid in self.trust.columns:
            trust_10 = self.trust[sid].rolling(10).sum().reindex(idx, method='ffill').fillna(0)
            trust_ok = trust_10 > 0
        else:
            trust_10 = pd.Series(0.0, index=idx)
            trust_ok = pd.Series(False, index=idx)

        big_chg, small_chg = self._inv_stats(sid, idx)  # 週頻 diff 後 reindex 的日頻變化量

        if sid in self.inv_pr_rank.columns:
            pr_series = (self.inv_pr_rank[sid] * 100).reindex(idx, method='ffill').fillna(0)
        else:
            pr_series = pd.Series(0.0, index=idx)

        hit_260 = (c >= high_260 * 0.995)
        low_vol_flag = candle_vol.fillna(99) < 10
        CONDS = {'創260高': hit_260, '低波動': low_vol_flag,
                 '融資健康': margin_ok, '營收9月高': rev_daily, '投信買超': trust_ok}
        last_status = '   '.join(f"{'✅' if CONDS[k].iloc[-1] else '❌'} {k}" for k in CONDS)

        fig = make_subplots(
            rows=6, cols=1, shared_xaxes=True,
            specs=[[{}], [{}], [{}], [{}], [{}], [{}]],
            row_heights=[0.40, 0.07, 0.11, 0.09, 0.16, 0.17],
            vertical_spacing=0.015,
            subplot_titles=['', '訊號條', '成交量', '融資維持率 (%)',
                            '月營收  YOY(柱) + MOM(線)  %', '集保：大戶持股比增減(柱)  ／  散戶持股比增減(線)  %'],
        )

        fig.add_trace(go.Candlestick(x=idx.astype(str), open=o, high=h, low=l, close=c, name='K棒',
                      showlegend=False, line_width=1,
                      increasing_line_color='#E74C3C', increasing_fillcolor='#E74C3C',
                      decreasing_line_color='#27AE60', decreasing_fillcolor='#27AE60'), row=1, col=1)
        for period, ckey in zip(self.ma_periods, self.ma_ckeys):
            fig.add_trace(go.Scatter(x=idx.astype(str), y=ma_dict[period], mode='lines',
                          name=f'MA{period}', line=dict(color=self.colors[ckey], width=1.4)), row=1, col=1)
        fig.add_trace(go.Scatter(x=idx.astype(str), y=high_260, mode='lines', name='260日高',
                      line=dict(color=self.colors['h260'], width=1.2, dash='dot')), row=1, col=1)

        dist_pct = (c.iloc[-1] / high_260.iloc[-1] - 1) * 100
        dist_clr = '#27AE60' if dist_pct >= -2 else ('#E74C3C' if dist_pct < -10 else '#F39C12')
        fig.add_annotation(text=last_status, xref='x domain', yref='y domain', x=0.01, y=0.97,
                           showarrow=False, font=dict(size=10, color='#1A252F'),
                           bgcolor='rgba(255,255,255,0.88)', bordercolor='rgba(180,180,180,0.65)',
                           borderwidth=1, borderpad=5, align='left', row=1, col=1)
        fig.add_annotation(
            text=f'<b>{c.iloc[-1]:.2f}</b>　260高 {high_260.iloc[-1]:.2f}　<b style="color:{dist_clr}">{dist_pct:+.1f}%</b>',
            xref='x domain', yref='y domain', x=0.99, y=0.97, showarrow=False,
            font=dict(size=11, color='#1A252F'), bgcolor='rgba(255,255,255,0.88)',
            bordercolor='rgba(180,180,180,0.65)', borderwidth=1, borderpad=5, align='right', row=1, col=1)

        for cname, cval in CONDS.items():
            cv = cval.reindex(idx).fillna(False)
            c_colors = [self.colors['cond_active'][cname] if v else self.colors['cond_inactive'] for v in cv]
            fig.add_trace(go.Scatter(x=idx.astype(str), y=[cname] * len(idx), mode='markers',
                          marker=dict(symbol='square', size=7, color=c_colors, line_width=0),
                          name=cname, showlegend=False), row=2, col=1)

        vol_colors = ['#C0392B' if cv >= ov else '#27AE60' for cv, ov in zip(c.fillna(0), o.fillna(0))]
        fig.add_trace(go.Bar(x=idx.astype(str), y=v, marker_color=vol_colors, showlegend=False, name='量'), row=3, col=1)
        fig.add_trace(go.Scatter(x=idx.astype(str), y=v.rolling(20).mean(), mode='lines',
                      name='量MA20', line=dict(color='#F39C12', width=1.3), showlegend=False), row=3, col=1)

        fig.add_trace(go.Scatter(x=idx.astype(str), y=margin_d, mode='lines', name='融資率',
                      line=dict(color='#2E86C1', width=1.6), showlegend=False), row=4, col=1)
        fig.add_trace(go.Scatter(x=idx.astype(str), y=margin_d.where(margin_ok), mode='none',
                      fill='tozeroy', fillcolor='rgba(46,134,193,0.12)', showlegend=False, hoverinfo='skip'), row=4, col=1)
        fig.add_hline(y=40, line_dash='dash', line_color='#E74C3C', line_width=1, annotation_text='40%', row=4, col=1)
        fig.add_hline(y=2, line_dash='dot', line_color='#717D7E', line_width=1, annotation_text='2%', row=4, col=1)

        yoy_cv = [self.colors['yoy_pos'] if val > 0 else self.colors['yoy_neg'] for val in yoy_d]
        fig.add_trace(go.Bar(x=idx.astype(str), y=yoy_d, marker_color=yoy_cv, name='YOY%', opacity=0.75), row=5, col=1)
        fig.add_trace(go.Scatter(x=idx.astype(str), y=mom_d, mode='lines', name='MOM%',
                      line=dict(color=self.colors['mom_line'], width=2.2, dash='dot')), row=5, col=1)
        fig.add_hline(y=0, line_color='rgba(100,100,100,0.5)', line_width=1, row=5, col=1)
        fig.add_annotation(text=f'YOY {yoy_d.iloc[-1]:+.1f}%   MOM {mom_d.iloc[-1]:+.1f}%',
                           xref='x5 domain', yref='y5 domain', x=0.01, y=0.93, showarrow=False,
                           font=dict(size=10, color='#1A252F'), bgcolor='rgba(255,255,255,0.80)', align='left')

        big_cv = ['#E74C3C' if val > 0 else '#27AE60' for val in big_chg]  # 紅=大戶加碼 綠=大戶減碼
        fig.add_trace(go.Bar(x=idx.astype(str), y=big_chg, marker_color=big_cv, name='大戶增減', opacity=0.85), row=6, col=1)
        fig.add_trace(go.Scatter(x=idx.astype(str), y=small_chg, mode='lines', name='散戶增減',
                      line=dict(color='#27AE60', width=1.8, dash='dot')), row=6, col=1)
        fig.add_hline(y=0, line_color='rgba(100,100,100,0.6)', line_width=1.5, row=6, col=1)

        br_last  = f'{big_chg.iloc[-1]:+.2f}%' if pd.notna(big_chg.iloc[-1]) else 'N/A'
        ppl_last = f'{small_chg.iloc[-1]:+.2f}%' if pd.notna(small_chg.iloc[-1]) else 'N/A'
        pr_last  = f'{pr_series.iloc[-1]:.0f}'

        fig.add_annotation(text=f'大戶4週增減 {br_last}   散戶4週增減 {ppl_last}   籌碼強弱 PR {pr_last}',
                           xref='x6 domain', yref='y6 domain', x=0.01, y=0.93, showarrow=False,
                           font=dict(size=10, color='#1A252F'), bgcolor='rgba(255,255,255,0.80)', align='left')

        date_str = str(idx[-1])[:10]
        fig.update_layout(
            title=dict(
                text=(
                    lambda co_name, strats: (
                        f'<b>&nbsp;{sid}&nbsp;&nbsp;{co_name.strip()}</b>'
                        f'&nbsp;&nbsp;&nbsp;│&nbsp;&nbsp;&nbsp;近 {self.days} 日'
                        f'&nbsp;&nbsp;&nbsp;│&nbsp;&nbsp;&nbsp;{date_str}'
                        + (f'&nbsp;&nbsp;&nbsp;│&nbsp;&nbsp;&nbsp;<span style="color:#E67E22">{" · ".join(strats)}</span>' if strats else '')
                    )
                )(self.company_map.get(sid, ''), self.stock_strategies.get(sid, [])),
                font=dict(size=16, color='#1A252F', family='Arial'), x=0.0),
            xaxis_rangeslider_visible=False, height=1020, margin=dict(l=65, r=65, t=55, b=30),
            legend=dict(orientation='h', y=1.042, x=0, font=dict(size=11, color='#2C3E50'), bgcolor='rgba(255,255,255,0)'),
            plot_bgcolor=self.colors['panel_bg'], paper_bgcolor=self.colors['bg'],
            barmode='overlay',
        )

        fig.update_yaxes(showgrid=False, zeroline=False, tickfont=dict(size=9, color='#555'), row=2, col=1)
        for r in [1, 3, 4, 5]:
            fig.update_yaxes(showgrid=True, gridcolor=self.colors['grid'], gridwidth=1,
                             zeroline=False, tickfont=dict(size=10), row=r, col=1)
        fig.update_yaxes(title_text='增減 %', title_font=dict(size=9, color='#1A252F'),
                         tickfont=dict(size=10, color='#1A252F'), showgrid=True,
                         gridcolor=self.colors['grid'], gridwidth=1, zeroline=False, row=6, col=1)
        for i in range(1, 7):
            fig.update_xaxes(type='category', tickangle=-45, nticks=12, showgrid=True,
                             gridcolor=self.colors['grid'], tickfont=dict(size=9, color='#666'), row=i, col=1)

        summary = (f"  收盤 {c.iloc[-1]:.2f}  260高 {high_260.iloc[-1]:.2f}  "
                   f"距離 {(c.iloc[-1]/high_260.iloc[-1]-1)*100:+.1f}%\n"
                   f"  {last_status}\n"
                   f"  融資 {margin_d.iloc[-1]:.1f}%  波動 {candle_vol.iloc[-1]:.1f}%\n"
                   f"  YOY {yoy_d.iloc[-1]:+.1f}%  MOM {mom_d.iloc[-1]:+.1f}%\n"
                   f"  大戶增減 {br_last}  散戶增減 {ppl_last}  "
                   f"投信10日 {trust_10.iloc[-1]:+.0f}張  籌碼PR {pr_last}")
        return fig, date_str, summary

    def plot_stock(self, sid):
        """顯示單檔圖表（互動模式，不儲存）"""
        fig, date_str, summary = self._build_fig(sid)
        fig.show()
        co_name = self.company_map.get(sid, '')
        print(f"{'─'*62}")
        print(f"  {sid}  {co_name}  {date_str}")
        print(summary)
        print()

    def run(self, save_html=True, save_pdf=False):
        """執行全部股票，可同時輸出合併 HTML 與合併 PDF"""
        import datetime
        date_tag = datetime.date.today().strftime('%Y-%m-%d')
        figs = []

        for sid in self.sid_exist:
            fig, date_str, summary = self._build_fig(sid)
            fig.show()
            co_name = self.company_map.get(sid, '')
            print(f"{'─'*62}")
            print(f"  {sid}  {co_name}  {date_str}")
            print(summary)
            print()
            figs.append((sid, fig))

        if not figs:
            print("沒有可輸出的股票")
            return

        # ── 合併 HTML（所有圖表串在同一頁，頂部有錨點目錄）──
        if save_html:
            html_path = os.path.join(self.output_dir, f'裸K看盤_{date_tag}.html')
            def _toc_tag(sid):
                co = self.company_map.get(sid, '')
                strats = self.stock_strategies.get(sid, [])
                strat_span = (
                    f'<span style="color:#E67E22;font-size:11px;margin-left:3px">'
                    + ' · '.join(strats) + '</span>'
                ) if strats else ''
                return (
                    f'<a href="#stock-{sid}" style="display:inline-block;margin:3px 5px;'
                    f'padding:3px 9px;background:#f0f4f8;border-radius:5px;color:#2E86C1;'
                    f'font-size:13px;text-decoration:none;white-space:nowrap;">'
                    f'<b>{sid}</b>&nbsp;{co}{strat_span}</a>'
                )
            toc_items = ''.join(_toc_tag(sid) for sid, _ in figs)
            toc_html = (
                f'<div style="position:sticky;top:0;z-index:999;background:#fff;'
                f'border-bottom:1px solid #ddd;padding:8px 16px;'
                f'display:flex;flex-wrap:wrap;align-items:center;gap:2px;">'
                f'<b style="font-size:13px;color:#555;white-space:nowrap;">股票目錄：</b>{toc_items}</div>'
            )
            blocks = []
            for i, (sid, fig) in enumerate(figs):
                inner = fig.to_html(full_html=False,
                                    include_plotlyjs='cdn' if i == 0 else False)
                blocks.append(f'<div id="stock-{sid}" style="margin-bottom:32px;">{inner}</div>')

            full_html = (
                '<!DOCTYPE html><html><head><meta charset="utf-8">'
                f'<title>裸K看盤 {date_tag}</title></head><body>'
                f'{toc_html}{"".join(blocks)}</body></html>'
            )
            with open(html_path, 'w', encoding='utf-8') as f:
                f.write(full_html)
            print(f'[HTML] {html_path}  （共 {len(figs)} 檔）')

        # ── 合併 PDF（每頁一支股票）──
        if save_pdf:
            pdf_path = os.path.join(self.output_dir, f'裸K看盤_{date_tag}.pdf')
            try:
                import io
                from pypdf import PdfWriter
                writer = PdfWriter()
                for sid, fig in figs:
                    buf = io.BytesIO()
                    fig.write_image(buf, format='pdf', width=1400, height=1020)
                    buf.seek(0)
                    writer.append(buf)
                with open(pdf_path, 'wb') as f:
                    writer.write(f)
                print(f'[PDF]  {pdf_path}  （共 {len(figs)} 頁）')
            except ImportError:
                print('[PDF] 缺少套件，請執行：uv add kaleido pypdf')
            except Exception as e:
                print(f'[PDF] 輸出失敗：{e}')


# 使用方式
if __name__ == "__main__":
    analyzer = StockAnalyzer()
    # save_html=True  → 所有股票合成一個 HTML（預設開啟）
    # save_pdf=True   → 所有股票合成一個多頁 PDF（需：uv add kaleido pypdf）
    analyzer.run(save_html=True, save_pdf=True)
